In [4]:
import pandas as pd
import numpy as np
import os
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy.stats import shapiro
import matplotlib.pyplot as plt
import seaborn as sns


In [5]:
path_data_clean = "../data_clean/"

file_data_bersih = os.path.join(path_data_clean, "dataset_final_2021-2024.csv")


df = pd.read_csv(file_data_bersih)
df.head()

,Wilayah,Tahun,P1,UHH,HLS,RLS,Pengeluaran,TPT,Kepadatan
0,Kabupaten Cilacap,2021,1.48,73.90,12.63,7.09,10534,9.97,924
1,Kabupaten Banyumas,2021,2.35,73.80,13.03,7.63,11546,6.05,1340
2,Kabupaten Purbalingga,2021,2.10,73.21,12.00,7.25,10032,6.05,1487
3,Kabupaten Banjarnegara,2021,2.97,74.28,11.63,6.75,9407,5.86,1003
4,Kabupaten Kebumen,2021,3.24,73.55,13.35,7.55,9028,6.03,1124


In [6]:
print(df.shape)

(140, 9)


Data terdiri dari 105 baris, 8 kolom

In [7]:
print("Jumlah data duplikat: ", df.duplicated().sum())

Jumlah data duplikat:  0


In [8]:
print(df.isna().sum())

Wilayah        0
Tahun          0
P1             0
UHH            0
HLS            0
RLS            0
Pengeluaran    0
TPT            0
Kepadatan      0
dtype: int64


In [9]:
print(df.describe())

             Tahun          P1         UHH         HLS       RLS  \
count   140.000000  140.000000  140.000000  140.000000  140.0000   
mean   2022.500000    1.590643   75.201714   13.031000    8.1740   
std       1.122048    0.674892    1.810811    0.913436    1.2933   
min    2021.000000    0.470000   69.540000   11.630000    6.2200   
25%    2021.750000    1.020000   74.167500   12.450000    7.2975   
50%    2022.500000    1.525000   75.000000   12.900000    7.8050   
75%    2023.250000    1.980000   76.470000   13.360000    8.8575   
max    2024.000000    3.410000   78.260000   15.570000   11.4800   

        Pengeluaran         TPT     Kepadatan  
count    140.000000  140.000000    140.000000  
mean   11790.107143    5.151000   2112.557143  
std     1850.027076    1.875406   2399.331982  
min     8573.000000    1.760000    461.000000  
25%    10512.000000    3.800000    935.750000  
50%    11413.500000    4.955000   1182.000000  
75%    12772.500000    6.052500   1846.000000  
max

PENGERJAAN

STEP 1 — Pisahkan data
- Train = tahun 2021 - 2023
- Test = tahun 2024

In [10]:
df_train = df[df["Tahun"].isin([2021, 2022, 2023])].copy()
df_test  = df[df["Tahun"] == 2024].copy()

print(df_train.shape, df_test.shape)

(105, 9) (35, 9)


STEP 2 — Uji signifikansi sederhana (satu-satu hubungan)

Regress:
- P1 ~ UHH
- P1 ~ HLS
- P1 ~ RLS
- P1 ~ Pengeluaran
- P1 ~ TPT
- P1 ~ Kepadatan

Tujuan uji ini: melihat variabel mana yang memiliki hubungan jelas dengan Y.

In [11]:
def simple_regression(df, x):
    X = sm.add_constant(df[[x]])
    y = df["P1"]
    model = sm.OLS(y, X).fit()
    print(f"\n===== P1 ~ {x} =====")
    print(model.summary())

for col in ["UHH", "HLS", "RLS", "Pengeluaran", "TPT", "Kepadatan"]:
    simple_regression(df_train, col)


===== P1 ~ UHH =====
                            OLS Regression Results                            
Dep. Variable:                     P1   R-squared:                       0.355
Model:                            OLS   Adj. R-squared:                  0.348
Method:                 Least Squares   F-statistic:                     56.63
Date:                Fri, 05 Dec 2025   Prob (F-statistic):           2.06e-11
Time:                        11:25:13   Log-Likelihood:                -86.734
No. Observations:                 105   AIC:                             177.5
Df Residuals:                     103   BIC:                             182.8
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         18.7358      2.2

STEP 3 — Bangun model regresi linier berganda (X1…X6)

Lakukan:
- uji F
- uji t
- cek VIF
- backward elimination kalau ada variabel tidak signifikan
- pilih model final

In [12]:
X_cols = ["UHH","HLS","RLS","Pengeluaran","TPT","Kepadatan"]
X_train = sm.add_constant(df_train[X_cols])
y_train = df_train["P1"]

model_full = sm.OLS(y_train, X_train).fit()
print(model_full.summary())


                            OLS Regression Results                            
Dep. Variable:                     P1   R-squared:                       0.438
Model:                            OLS   Adj. R-squared:                  0.404
Method:                 Least Squares   F-statistic:                     12.74
Date:                Fri, 05 Dec 2025   Prob (F-statistic):           1.37e-10
Time:                        11:25:17   Log-Likelihood:                -79.455
No. Observations:                 105   AIC:                             172.9
Df Residuals:                      98   BIC:                             191.5
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          13.7361      3.866      3.553      

In [13]:
# cek uji multikolinearitas (VIF)
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

vif_df = pd.DataFrame()
vif_df["feature"] = X_cols
vif_df["VIF"] = [variance_inflation_factor(df_train[X_cols].values, i)
                 for i in range(len(X_cols))]

vif_df


,feature,VIF
0,UHH,597.255441
1,HLS,1750.062966
2,RLS,534.719763
3,Pengeluaran,135.339262
4,TPT,15.226260
5,Kepadatan,5.855145


In [14]:
# Backward elimination jika ada variabel tidak signifikan
def backward_elimination(X, y, alpha=0.05):
    X = sm.add_constant(X)
    while True:
        model = sm.OLS(y, X).fit()
        pvals = model.pvalues.drop("const")
        max_p = pvals.max()
        if max_p > alpha:
            drop_var = pvals.idxmax()
            print("Drop:", drop_var, "(p =", max_p, ")")
            X = X.drop(columns=[drop_var])
        else:
            break
    return model, X.columns

model_final, selected_features = backward_elimination(df_train[X_cols], y_train)
print(selected_features)
print(model_final.summary())


Drop: Kepadatan (p = 0.796405838152286 )
Drop: HLS (p = 0.6318162657846371 )
Drop: Pengeluaran (p = 0.46089606709343534 )
Drop: TPT (p = 0.37942852554022677 )
Index(['const', 'UHH', 'RLS'], dtype='object')
                            OLS Regression Results                            
Dep. Variable:                     P1   R-squared:                       0.429
Model:                            OLS   Adj. R-squared:                  0.418
Method:                 Least Squares   F-statistic:                     38.34
Date:                Fri, 05 Dec 2025   Prob (F-statistic):           3.83e-13
Time:                        11:25:34   Log-Likelihood:                -80.304
No. Observations:                 105   AIC:                             166.6
Df Residuals:                     102   BIC:                             174.6
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
    

STEP 4 — Prediksi tahun 2024 (test set)

- Ambil model final dari train (2021–2023)
- Prediksi P1 tahun 2024
- Hitung MAE, RMSE, MAPE

In [15]:
use_cols = [c for c in selected_features if c != "const"]

X_test = sm.add_constant(df_test[use_cols])
df_test["P1_pred"] = model_final.predict(X_test)

df_test[["Wilayah", "Tahun", "P1", "P1_pred"]]


,Wilayah,Tahun,P1,P1_pred
105,Kabupaten Cilacap,2024,1.59,1.840160
106,Kabupaten Banyumas,2024,2.09,1.754395
107,Kabupaten Purbalingga,2024,2.10,1.964472
108,Kabupaten Banjarnegara,2024,2.38,1.946333
109,Kabupaten Kebumen,2024,2.35,1.806649
110,Kabupaten Purworejo,2024,1.08,1.457958
111,Kabupaten Wonosobo,2024,2.41,2.198236
112,Kabupaten Magelang,2024,1.23,1.759394
113,Kabupaten Boyolali,2024,1.56,1.457066
114,Kabupaten Klaten,2024,1.46,1.112582


In [16]:
y_true = df_test["P1"]
y_pred = df_test["P1_pred"]

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print("MAE:", mae)
print("RMSE:", rmse)
print("MAPE:", mape)


MAE: 0.4223936854499217
RMSE: 0.5122841275654858
MAPE: 33.478875283264856
